# Transaction Anomaly Detection

trying to flag suspicious transactions using a small neural net.  
labels come from the intelligence_alerts.csv — any txn that appears there is treated as anomalous.

data: `transactions.csv` (43k rows), `intelligence_alerts.csv`

In [1]:
import csv
import json
import math
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from collections import defaultdict
from datetime import datetime
from pathlib import Path

print('TF:', tf.__version__)

TF: 2.16.1


In [2]:
# paths — notebook lives in backend/models/, data is one level up
DATA = Path('..') / 'data'

def load_csv(fname):
    with open(DATA / fname, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

txns    = load_csv('transactions.csv')
alerts  = load_csv('intelligence_alerts.csv')
vendors = load_csv('vendors.csv')
ddos    = load_csv('ddo_accounts.csv')

print(f'transactions : {len(txns)}')
print(f'alerts       : {len(alerts)}')
print(f'vendors      : {len(vendors)}')
print(f'ddos         : {len(ddos)}')

transactions : 43004
alerts       : 3407
vendors      : 500
ddos         : 284


In [3]:
# quick look at the columns
print('txn cols    :', list(txns[0].keys()))
print('alert cols  :', list(alerts[0].keys()))
print('sample txn  :', txns[0])

txn cols    : ['id', 'sanction_id', 'ddo_code', 'vendor_id', 'amount', 'purpose_category', 'release_date', 'utilization_date', 'status']
alert cols  : ['alert_id', 'tx_id', 'dept_id', 'alert_type', 'confidence_score', 'description', 'timestamp']
sample txn  : {'id': '1', 'sanction_id': 'SAN000001', 'ddo_code': 'DDO0001', 'vendor_id': '63', 'amount': '2118000', 'purpose_category': 'Welfare', 'release_date': '2022-12-31', 'utilization_date': '2023-01-15', 'status': 'utilized'}


In [4]:
# which txns are flagged?
flagged_ids = set()
for a in alerts:
    tid = a.get('transaction_id', '').strip()
    if tid:
        flagged_ids.add(tid)

print(f'flagged transaction ids: {len(flagged_ids)}')
print(f'anomaly rate           : {len(flagged_ids)/len(txns)*100:.2f}%')

flagged transaction ids: 0
anomaly rate           : 0.00%


In [5]:
# build some lookup tables we'll need for features
# vendor risk_score is already 0-1 in the CSV, use it directly
vendor_risk = {v['id']: float(v.get('risk_score', 0)) for v in vendors}

ddo_tx_counts   = defaultdict(int)
ddo_amounts     = defaultdict(list)
for t in txns:
    ddo_tx_counts[t['ddo_code']] += 1
    try:
        ddo_amounts[t['ddo_code']].append(float(t['amount']))
    except:
        pass

MAX_DDO_TX  = max(ddo_tx_counts.values())
ddo_avg_amt = {k: np.mean(v) for k, v in ddo_amounts.items()}
MAX_AVG_AMT = max(ddo_avg_amt.values())

print('max txns per ddo :', MAX_DDO_TX)
print('max avg amt      :', round(MAX_AVG_AMT, 2))

max txns per ddo : 202
max avg amt      : 1708277.78


In [6]:
# amount distribution — rough check before normalising
amounts = [float(t['amount']) for t in txns]
print(f'min={min(amounts):.0f}  max={max(amounts):.0f}  mean={np.mean(amounts):.0f}  median={np.median(amounts):.0f}')

# log transform will help here
MAX_LOG = math.log1p(max(amounts))

min=40000  max=4498000  mean=1469036  median=1284500


## Feature Engineering

10 features per transaction. keeping it simple — amount stats, timing, category, vendor risk, ddo volume.

In [7]:
# z-score for amount (just using global mean/std, good enough)
amt_mean = float(np.mean(amounts))
amt_std  = float(np.std(amounts)) or 1.0

# category encoding
categories = sorted({t['purpose_category'] for t in txns})
cat_to_idx = {c: i for i, c in enumerate(categories)}
MAX_CAT    = max(len(categories) - 1, 1)

print(f'{len(categories)} unique categories')

10 unique categories


In [8]:
def extract_features(t):
    amt = float(t['amount'])

    # amount
    amt_log_norm  = math.log1p(amt) / MAX_LOG
    amt_z         = (amt - amt_mean) / amt_std
    amt_z_norm    = (amt_z - (-5)) / 10  # rough clip to [0,1]
    amt_z_norm    = max(0.0, min(1.0, amt_z_norm))

    # timing
    try:
        rd = t['release_date']; ud = t['utilization_date']
        from datetime import date
        r = date.fromisoformat(rd); u = date.fromisoformat(ud)
        delay = (u - r).days
        util_delay_norm = min(max(delay / 365.0, 0.0), 1.0)
        month = r.month
        is_q4 = 1.0 if month >= 10 else 0.0
        is_q1 = 1.0 if month <= 3  else 0.0
        util_missing = 0.0
    except:
        util_delay_norm = 0.5
        is_q4 = 0.0; is_q1 = 0.0
        util_missing = 1.0

    # categorical
    cat_enc = cat_to_idx.get(t['purpose_category'], 0) / MAX_CAT
    v_risk  = float(vendor_risk.get(t['vendor_id'], 0))

    # ddo-level stats
    dco = t['ddo_code']
    ddo_vol  = ddo_tx_counts.get(dco, 0) / MAX_DDO_TX
    ddo_avg  = ddo_avg_amt.get(dco, amt_mean) / MAX_AVG_AMT

    return [
        amt_log_norm, amt_z_norm, util_delay_norm,
        is_q4, is_q1, cat_enc, v_risk,
        ddo_vol, ddo_avg, util_missing
    ]

# test on one row
print(extract_features(txns[0]))

[0.9508353485979054, 0.565115075517493, 0.0410958904109589, 1.0, 0.0, 1.0, 0.37, 0.7871287128712872, 0.8723854802457144, 0.0]


In [9]:
np.random.seed(42)
tf.random.set_seed(42)

X, y = [], []
for t in txns:
    X.append(extract_features(t))
    y.append(1 if t['id'] in flagged_ids else 0)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

print(f'X shape: {X.shape}')
print(f'positives (anomaly): {int(y.sum())} / {len(y)}')

X shape: (43004, 10)
positives (anomaly): 0 / 43004


## Model Training

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test  = sc.transform(X_test)

n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
w     = n_neg / max(n_pos, 1)
print(f'class weight: 1={w:.1f}x  (neg={n_neg}, pos={n_pos})')

class weight: 1=34403.0x  (neg=34403, pos=0)


In [11]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.1),
    layers.Dense(1, activation='sigmoid')
], name='anomaly_detector')

model.summary()

Model: "anomaly_detector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,545 (49.00 KB)

 Trainable params: 12,161 (47.50 KB)

 Non-trainable params: 384 (1.50 KB)

In [12]:
model.compile(
    optimizer=keras.optimizers.Adam(3e-4),
    loss='binary_crossentropy',
    metrics=[
        keras.metrics.AUC(name='auc'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
    ]
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_auc', patience=10,
                                  restore_best_weights=True, mode='max'),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                      patience=5, min_lr=1e-6),
]

history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=256,
    class_weight={0: 1.0, 1: w},
    callbacks=callbacks,
    verbose=1
)

Epoch 1/100


  1/115 ━━━━━━━━━━━━━━━━━━━━ 3:29 2s/step - auc: 0.0000e+00 - loss: 0.4840 - precision: 0.0000e+00 - recall: 0.0000e+00

 21/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - auc: 0.0000e+00 - loss: 0.4375 - precision: 0.0000e+00 - recall: 0.0000e+00 

 44/115 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.0000e+00 - loss: 0.3920 - precision: 0.0000e+00 - recall: 0.0000e+00

 68/115 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.0000e+00 - loss: 0.3537 - precision: 0.0000e+00 - recall: 0.0000e+00

...

 61/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - auc: 0.0000e+00 - loss: 7.9389e-04 - precision: 0.0000e+00 - recall: 0.0000e+00

 74/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - auc: 0.0000e+00 - loss: 7.8793e-04 - precision: 0.0000e+00 - recall: 0.0000e+00

 92/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - auc: 0.0000e+00 - loss: 7.8143e-04 - precision: 0.0000e+00 - recall: 0.0000e+00

108/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - auc: 0.0000e+00 - loss: 7.7653e-04 - precision: 0.0000e+00 - recall: 0.0000e+00

115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - auc: 0.0000e+00 - loss: 7.4268e-04 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_auc: 0.0000e+00 - val_loss: 7.0770e-05 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 3.0000e-04


## Evaluation

In [13]:
y_prob = model.predict(X_test, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)
auc  = roc_auc_score(y_test, y_prob)
cm   = confusion_matrix(y_test, y_pred)

print(f'Accuracy  {acc:.4f}')
print(f'Precision {prec:.4f}')
print(f'Recall    {rec:.4f}')
print(f'F1        {f1:.4f}')
print(f'AUC-ROC   {auc:.4f}')
print(f'CM:\n{cm}')

Accuracy  1.0000
Precision 0.0000
Recall    0.0000
F1        0.0000
AUC-ROC   nan
CM:
[[8601]]


C:\Users\karan\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
C:\Users\karan\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


In [14]:
model.save('anomaly_detector.h5')

stats = {
    'model': 'anomaly_detector',
    'framework': f'TensorFlow {tf.__version__}',
    'trained_at': datetime.utcnow().isoformat() + 'Z',
    'samples': int(len(y)),
    'anomaly_rate_pct': round(float(y.mean()) * 100, 2),
    'features': [
        'amt_log_norm', 'amt_z_norm', 'util_delay_norm',
        'is_q4', 'is_q1', 'cat_enc', 'vendor_risk',
        'ddo_tx_vol', 'ddo_avg_amt', 'util_missing'
    ],
    'scaler_mean':  sc.mean_.tolist(),
    'scaler_scale': sc.scale_.tolist(),
    'performance': {
        'accuracy':  round(acc,  4),
        'precision': round(prec, 4),
        'recall':    round(rec,  4),
        'f1':        round(f1,   4),
        'auc_roc':   round(auc,  4),
    },
    'confusion_matrix': cm.tolist()
}

with open('model_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print('saved anomaly_detector.h5  +  model_stats.json')

saved anomaly_detector.h5  +  model_stats.json
